# Lab 3

Importy:

In [2]:
from collections import defaultdict, Counter
import math
from statistics import pstdev

In [ ]:
def read_file(filename):
    with open(filename, "r") as f:
        return f.read()
    
def count_letters(text, order = 1):
    letter_counts = defaultdict(Counter)
    for i in range(len(text) - order):
        context = text[i:i+order]
        letter = text[i+order]
        letter_counts[context][letter] += 1
    return letter_counts


def count_words(text, order = 1):
    word_counts = defaultdict(Counter)
    words = text.split()
    for i in range(len(words) - order):
        context = tuple(words[i:i+order])
        word = words[i+order]
        word_counts[context][word] += 1
    return word_counts

def create_probability_list(counts, total_count, order = 1):
    probability_list = defaultdict()
    omega = total_count - order
    for context, counter in counts.items():
        for item, count in counter.items():
            probability = count / omega
            probability_list[str(context) + item] = probability
    return probability_list

def count_entropy(probability_list):
    entropy = 0
    for prob in probability_list.values():
        if prob > 0:
            entropy -= prob * math.log2(prob)
    return entropy

def count_conditional_entropy(counts):
    conditional_entropy = 0
    total_count = sum(sum(counter.values()) for counter in counts.values())
    for counter in counts.values():
        context_count = sum(counter.values())
        for count in counter.values():
            if count > 0 and context_count > 0:
                p_xy = count / total_count
                p_y_given_x = count / context_count
                conditional_entropy -= p_xy * math.log2(p_y_given_x)
    return conditional_entropy

Jaka będzie entropia przyblizenia zerowego rzędu dla przyblizenia języka
angielskiego generowanego na poziomie znaków? Przypomnienie: 26 liter +
cyfry + spacja, mające jednakowe prawdopodobieństwo wystąpienia.
Policz entropię występowania znaków w języku angielskim na podstawie
kodu z pierwszych zajęć.

In [3]:
N = 26 + 10 + 1  # 26 letters + 10 digits + 1 space
p_i = 1 / N
entropy_uniform = -N * p_i * math.log2(p_i)
print(f"Entropy of uniform distribution for EncodingWarning: {entropy_uniform:.4f} bits per symbol")

Entropy of uniform distribution for EncodingWarning: 5.2095 bits per symbol


Wylicz entropie znaków i słów oraz ich entropie warunkowe kolejnych rządów dla próbki języka angielskiego (plik norm_wiki_en.txt). 

In [6]:
text = read_file("norm_wiki_en.txt")

for o in range(6):
    print(f"Order: {o}")
    letter_counts = count_letters(text, order = o)
    word_counts = count_words(text, order = o)

    prob_letters = create_probability_list(letter_counts, len(text), order = o)
    prob_words = create_probability_list(word_counts, len(text.split()), order = o)

    entropy_letters = count_entropy(prob_letters)
    entropy_words = count_entropy(prob_words)

    print(f"Entropy of letters: {entropy_letters:.4f} bits")
    print(f"Entropy of words: {entropy_words:.4f} bits")

    conditional_entropy_letters = count_conditional_entropy(letter_counts)
    conditional_entropy_words = count_conditional_entropy(word_counts)

    print(f"Conditional Entropy of letters: {conditional_entropy_letters:.4f} bits")
    print(f"Conditional Entropy of words: {conditional_entropy_words:.4f} bits")

Order: 0
Entropy of letters: 4.2882 bits
Entropy of words: 11.5440 bits
Conditional Entropy of letters: 4.2882 bits
Conditional Entropy of words: 11.5440 bits
Order: 1
Entropy of letters: 7.8048 bits
Entropy of words: 17.9332 bits
Conditional Entropy of letters: 3.5166 bits
Conditional Entropy of words: 6.3892 bits
Order: 2
Entropy of letters: 10.8231 bits
Entropy of words: 20.1096 bits
Conditional Entropy of letters: 3.0183 bits
Conditional Entropy of words: 2.1765 bits
Order: 3
Entropy of letters: 13.3047 bits
Entropy of words: 20.5943 bits
Conditional Entropy of letters: 2.4816 bits
Conditional Entropy of words: 0.4847 bits
Order: 4
Entropy of letters: 15.3259 bits
Entropy of words: 20.7040 bits
Conditional Entropy of letters: 2.0212 bits
Conditional Entropy of words: 0.1097 bits
Order: 5
Entropy of letters: 16.9983 bits
Entropy of words: 20.7347 bits
Conditional Entropy of letters: 1.6724 bits
Conditional Entropy of words: 0.0308 bits


Następnie wylicz entropie znaków i słów oraz ich entropie warunkowe kolejnych rzędów dla próbki języka łacińskiego (plik norm_wiki_lo.txt). (2pt)
Mozesz równiez dokonać analizy dla próbek innych języków:
- esperanto (plik norm_wiki_eo.txt),
- estońskiego (plik norm_wiki_et.txt),
- somalijski (plik norm_wiki_so.txt),
- haitański (plik norm_wiki_ht.txt),
- navaho (plik norm_wiki_nv.txt),

In [22]:
def count_all_entropies(text, max_order = 5):
    entropies = []
    for o in range(max_order + 1):
        letter_counts = count_letters(text, order = o)
        word_counts = count_words(text, order = o)

        prob_letters = create_probability_list(letter_counts, len(text), order = o)
        prob_words = create_probability_list(word_counts, len(text.split()), order = o)

        entropy_letters = count_entropy(prob_letters)
        entropy_words = count_entropy(prob_words)

        conditional_entropy_letters = count_conditional_entropy(letter_counts)
        conditional_entropy_words = count_conditional_entropy(word_counts)

        entropies.append((o, entropy_letters, entropy_words, conditional_entropy_letters, conditional_entropy_words))
    return entropies

In [23]:
texts = ['norm_wiki_la.txt', 'norm_wiki_eo.txt', 'norm_wiki_et.txt', 'norm_wiki_ht.txt', 'norm_wiki_nv.txt', 'norm_wiki_so.txt']
output = {}

for filename in texts:
    print(f"Processing file: {filename}")
    text = read_file(filename)
    entropies = count_all_entropies(text, max_order = 5)
    output[filename] = entropies

Processing file: norm_wiki_la.txt
Processing file: norm_wiki_eo.txt
Processing file: norm_wiki_et.txt
Processing file: norm_wiki_ht.txt
Processing file: norm_wiki_nv.txt
Processing file: norm_wiki_so.txt


In [ ]:
for key, value in output.items():
    print(f"File: {key}")
    for order, entropy_letters, entropy_words, conditional_entropy_letters, conditional_entropy_words in value:
        print(f"Order: {order}, Entropy Letters: {entropy_letters:.4f}, Entropy Words: {entropy_words:.4f}, Conditional Entropy Letters: {conditional_entropy_letters:.4f}, Conditional Entropy Words: {conditional_entropy_words:.4f}")


File: norm_wiki_la.txt
Order: 0, Entropy Letters: 4.2282, Entropy Words: 11.9692, Conditional Entropy Letters: 4.2282, Conditional Entropy Words: 11.9692
Order: 1, Entropy Letters: 7.6784, Entropy Words: 16.3692, Conditional Entropy Letters: 3.4501, Conditional Entropy Words: 4.4000
Order: 2, Entropy Letters: 10.5019, Entropy Words: 17.5361, Conditional Entropy Letters: 2.8235, Conditional Entropy Words: 1.1669
Order: 3, Entropy Letters: 12.6539, Entropy Words: 17.9241, Conditional Entropy Letters: 2.1520, Conditional Entropy Words: 0.3880
Order: 4, Entropy Letters: 14.2967, Entropy Words: 18.1306, Conditional Entropy Letters: 1.6428, Conditional Entropy Words: 0.2065
Order: 5, Entropy Letters: 15.6094, Entropy Words: 18.2957, Conditional Entropy Letters: 1.3127, Conditional Entropy Words: 0.1651
File: norm_wiki_eo.txt
Order: 0, Entropy Letters: 4.1768, Entropy Words: 11.5605, Conditional Entropy Letters: 4.1768, Conditional Entropy Words: 11.5605
Order: 1, Entropy Letters: 7.5168, Ent

In [24]:
mean_entropy_letters = [0 for _ in range(6)]
mean_entropy_words = [0 for _ in range(6)]
mean_conditional_entropy_letters = [0 for _ in range(6)]
mean_conditional_entropy_words = [0 for _ in range(6)]

for key, value in output.items():
    for order, entropy_letters, entropy_words, conditional_entropy_letters, conditional_entropy_words in value:
        mean_entropy_letters[order] += entropy_letters
        mean_entropy_words[order] += entropy_words
        mean_conditional_entropy_letters[order] += conditional_entropy_letters
        mean_conditional_entropy_words[order] += conditional_entropy_words

for i in range(6):
    mean_entropy_letters[i] /= len(output)
    mean_entropy_words[i] /= len(output)
    mean_conditional_entropy_letters[i] /= len(output)
    mean_conditional_entropy_words[i] /= len(output)

print("Mean Entropy of letters for each order:")
for i in range(6):
    print(f"Order {i}: {mean_entropy_letters[i]:.4f} bits")

print("Mean Entropy of words for each order:")
for i in range(6):
    print(f"Order {i}: {mean_entropy_words[i]:.4f} bits")

print("Mean Conditional Entropy of letters for each order:")
for i in range(6):
    print(f"Order {i}: {mean_conditional_entropy_letters[i]:.4f} bits")

print("Mean Conditional Entropy of words for each order:")
for i in range(6):
    print(f"Order {i}: {mean_conditional_entropy_words[i]:.4f} bits")

Mean Entropy of letters for each order:
Order 0: 4.1061 bits
Order 1: 7.3823 bits
Order 2: 10.1015 bits
Order 3: 12.2378 bits
Order 4: 13.9185 bits
Order 5: 15.2716 bits
Mean Entropy of words for each order:
Order 0: 11.0547 bits
Order 1: 15.8609 bits
Order 2: 17.3935 bits
Order 3: 17.9366 bits
Order 4: 18.2145 bits
Order 5: 18.4009 bits
Mean Conditional Entropy of letters for each order:
Order 0: 4.1061 bits
Order 1: 3.2763 bits
Order 2: 2.7192 bits
Order 3: 2.1362 bits
Order 4: 1.6807 bits
Order 5: 1.3531 bits
Mean Conditional Entropy of words for each order:
Order 0: 11.0547 bits
Order 1: 4.8063 bits
Order 2: 1.5325 bits
Order 3: 0.5432 bits
Order 4: 0.2779 bits
Order 5: 0.1864 bits


Korzystając z zaobserwowanych wartości entropii warunkowej odpowiedź
na pytanie, czy następujące pliki zawierają język naturalny: (6pt) (po (1pt)
za kazy dobrze rozpoznany plik)
- sample0.txt,
- sample1.txt,
- sample2.txt,
- sample3.txt,
- sample4.txt,
- sample5.txt.

In [25]:
files = ['sample0.txt', 'sample1.txt', 'sample2.txt', 'sample3.txt', 'sample4.txt', 'sample5.txt']
samples = {}

for filename in files:
    print(f"Processing file: {filename}")
    text = read_file(filename)
    entropies = count_all_entropies(text, max_order = 5)
    entropy_letters_diff = [entropies[i][1] - mean_entropy_letters[i] for i in range(6)]
    entropy_words_diff = [entropies[i][2] - mean_entropy_words[i] for i in range(6)]
    conditional_entropy_letters_diff = [entropies[i][3] - mean_conditional_entropy_letters[i] for i in range(6)]
    conditional_entropy_words_diff = [entropies[i][4] - mean_conditional_entropy_words[i] for i in range(6)]
    samples[filename] = (entropy_letters_diff, entropy_words_diff, conditional_entropy_letters_diff, conditional_entropy_words_diff)

for key, value in samples.items():
    print(f"File: {key}")
    for i in range(6):
        print(f"Order {i}: Entropy Letters Diff: {value[0][i]:.4f}, Entropy Words Diff: {value[1][i]:.4f}, Conditional Entropy Letters Diff: {value[2][i]:.4f}, Conditional Entropy Words Diff: {value[3][i]:.4f}")

    
    

Processing file: sample0.txt
Processing file: sample1.txt
Processing file: sample2.txt
Processing file: sample3.txt
Processing file: sample4.txt
Processing file: sample5.txt
File: sample0.txt
Order 0: Entropy Letters Diff: 0.1670, Entropy Words Diff: -3.3059, Conditional Entropy Letters Diff: 0.1670, Conditional Entropy Words Diff: -3.3059
Order 1: Entropy Letters Diff: -0.1935, Entropy Words Diff: -0.6258, Conditional Entropy Letters Diff: -0.3604, Conditional Entropy Words Diff: 2.6801
Order 2: Entropy Letters Diff: -0.9123, Entropy Words Diff: 2.2484, Conditional Entropy Letters Diff: -0.7188, Conditional Entropy Words Diff: 2.8742
Order 3: Entropy Letters Diff: -1.5092, Entropy Words Diff: 2.3002, Conditional Entropy Letters Diff: -0.5969, Conditional Entropy Words Diff: 0.0519
Order 4: Entropy Letters Diff: -1.7514, Entropy Words Diff: 2.0344, Conditional Entropy Letters Diff: -0.2422, Conditional Entropy Words Diff: -0.2659
Order 5: Entropy Letters Diff: -1.7125, Entropy Words Di

In [ ]:
# 1) Odchylenia standardowe z referencji (output)
std_H_l = [0.0] * 6
std_H_w = [0.0] * 6
std_HC_l = [0.0] * 6
std_HC_w = [0.0] * 6

for o in range(6):
    vals_H_l = [rows[o][1] for rows in output.values()]
    vals_H_w = [rows[o][2] for rows in output.values()]
    vals_HC_l = [rows[o][3] for rows in output.values()]
    vals_HC_w = [rows[o][4] for rows in output.values()]

    std_H_l[o] = pstdev(vals_H_l) or 1e-12
    std_H_w[o] = pstdev(vals_H_w) or 1e-12
    std_HC_l[o] = pstdev(vals_HC_l) or 1e-12
    std_HC_w[o] = pstdev(vals_HC_w) or 1e-12

# 2) Klasyfikacja na podstawie wcześniej policzonych różnic (samples)
results = {}

for fname, (dH_l, dH_w, dHC_l, dHC_w) in samples.items():
    z_H_l = [abs(dH_l[o]) / std_H_l[o] for o in range(6)]
    z_H_w = [abs(dH_w[o]) / std_H_w[o] for o in range(6)]
    z_HC_l = [abs(dHC_l[o]) / std_HC_l[o] for o in range(6)]
    z_HC_w = [abs(dHC_w[o]) / std_HC_w[o] for o in range(6)]

    # większa waga dla entropii warunkowej
    score = (
        0.2 * sum(z_H_l) / 6 +
        0.2 * sum(z_H_w) / 6 +
        0.3 * sum(z_HC_l) / 6 +
        0.3 * sum(z_HC_w) / 6
    )

    if score < 2.0:
        label = "likely natural language"
    elif score < 3.0:
        label = "possibly natural language"
    else:
        label = "unlikely natural language"

    results[fname] = (score, label)

for fname, (score, label) in sorted(results.items(), key=lambda x: x[1][0]):
    print(f"{fname:12s} score={score:.3f} -> {label}")

sample1.txt  score=0.280 -> likely natural language
sample3.txt  score=0.786 -> likely natural language
sample2.txt  score=1.104 -> likely natural language
sample0.txt  score=1.447 -> likely natural language
sample5.txt  score=1.777 -> likely natural language
sample4.txt  score=2.948 -> possibly natural language
